In [1]:
# Cell 1: Imports & Setup
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.manifold import TSNE
from scipy.stats import wasserstein_distance, ks_2samp
import warnings
warnings.filterwarnings('ignore')

# Project paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'glioma_dataset.h5')
SPLIT_FILE = os.path.join(PROJECT_ROOT, 'data', 'splits', 'manual_splits.json')
FIGURES_DIR = os.path.join(PROJECT_ROOT, 'paper', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

from src.data.loader import GliomaDataLoader
from src.models.baselines import ApolloPipeline

# Publication-quality matplotlib settings
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

CLASS_NAMES = ['LGm-1', 'LGm-2', 'LGm-3', 'LGm-4', 'LGm-5', 'LGm-6']
print(f'Project root: {PROJECT_ROOT}')
print(f'Figures dir:  {FIGURES_DIR}')

SyntaxError: unterminated triple-quoted string literal (detected at line 207) (loader.py, line 191)

In [ ]:
# Cell 2: Load Data & Run TRTR Baseline
print('Loading tumor-only data...')
loader = GliomaDataLoader(
    data_path=DATA_PATH,
    split_file=SPLIT_FILE,
    memory_mapped=False,
    tumor_only=True
)
print(f'Total tumor spectra: {loader.X.shape}')
print(f'Class distribution: {np.bincount(loader.y_class)}')

In [ ]:
# Cell 3: Run full 5-fold TRTR baseline and collect predictions
trtr_results = []
all_y_true = []
all_y_pred = []

print('Running TRTR Baseline (Apollo: RF Top-20 + SVM)...')
print('=' * 60)

for fold_data in loader:
    fold_name = fold_data['fold_name']
    X_train, y_train = fold_data['train']
    X_val, y_val = fold_data['val']
    
    print(f'\n{fold_name}: Train={X_train.shape[0]}, Val={X_val.shape[0]}')
    print(f'  Held-out patients: {fold_data["val_sample_ids"]}')
    
    model = ApolloPipeline(n_features=20, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    
    acc = accuracy_score(y_val, y_pred)
    print(f'  Accuracy: {acc:.4f}')
    
    trtr_results.append({'fold': fold_name, 'accuracy': acc})
    all_y_true.extend(y_val)
    all_y_pred.extend(y_pred)

mean_acc = np.mean([r['accuracy'] for r in trtr_results])
std_acc = np.std([r['accuracy'] for r in trtr_results])
print(f'\n{"=" * 60}')
print(f'TRTR Mean Accuracy: {mean_acc:.4f} ± {std_acc:.4f}')

In [ ]:
# Cell 4: Figure 1 — TRTR Baseline Confusion Matrix
print('Generating Figure 1: TRTR Confusion Matrix...')

all_y_true_arr = np.array(all_y_true)
all_y_pred_arr = np.array(all_y_pred)

# Get unique classes present
unique_classes = np.sort(np.unique(np.concatenate([all_y_true_arr, all_y_pred_arr])))
class_labels = [CLASS_NAMES[c-1] if 1 <= c <= 6 else f'Class {c}' for c in unique_classes]

cm = confusion_matrix(all_y_true_arr, all_y_pred_arr, labels=unique_classes)
cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-8)

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
            xticklabels=class_labels, yticklabels=class_labels,
            linewidths=0.5, linecolor='gray', ax=ax,
            cbar_kws={'label': 'Normalized Frequency'})
ax.set_title(f'TRTR Baseline: Apollo Pipeline\n'
             f'Patient-Wise 5-Fold CV (Mean Acc: {mean_acc:.1%})', fontweight='bold')
ax.set_xlabel('Predicted Class')
ax.set_ylabel('True Class')

save_path = os.path.join(FIGURES_DIR, 'fig1_trtr_confusion_matrix.pdf')
plt.savefig(save_path)
plt.show()
print(f'Saved: {save_path}')

## CVAE Evaluation

Load the trained CVAE model and generate synthetic data for TSTR evaluation.

In [ ]:
# Cell 5: Load CVAE Model
import tensorflow as tf
from src.utils.config import ExperimentConfig
from src.models.cvae import CVAE

# Find the latest experiment folder
EXP_BASE = os.path.join(PROJECT_ROOT, 'experiments')
if os.path.exists(EXP_BASE):
    exp_folders = sorted([d for d in os.listdir(EXP_BASE) if d.startswith('CVAE')])
    if exp_folders:
        LATEST_EXP = exp_folders[-1]
        WEIGHTS_PATH = os.path.join(EXP_BASE, LATEST_EXP, 'best_weights.weights.h5')
        print(f'Found experiment: {LATEST_EXP}')
        print(f'Weights path: {WEIGHTS_PATH}')
    else:
        print('No CVAE experiments found. Run notebook 03 first.')
        WEIGHTS_PATH = None
else:
    print('No experiments directory found. Run notebook 03 first.')
    WEIGHTS_PATH = None

# Recreate exact config from training
config = ExperimentConfig(
    experiment_name='Paper_Eval',
    input_shape=[1738, 1],
    num_classes=7,
    latent_dim=16,
    batch_size=128,
    epochs=50,
    learning_rate=0.0001,
    encoder={
        'filters': [32, 64, 128], 'kernels': [7, 7, 5], 'strides': [2, 2, 2],
        'use_skip_gap': False, 'use_global_pooling': False
    },
    decoder={
        'initial_filters': 128, 'filters': [64, 32, 16], 'kernels': [5, 7, 7],
        'strides': [2, 2, 2], 'use_dense_projection': True
    },
    loss_weights={'beta': 0.5, 'alpha': 2.0}
)

cvae = None
if WEIGHTS_PATH and os.path.exists(WEIGHTS_PATH):
    try:
        cvae = CVAE(config)
        cvae.build(input_shape=[(None, 1738, 1), (None, 7)])
        cvae.load_weights(WEIGHTS_PATH)
        print('CVAE loaded successfully.')
    except Exception as e:
        print(f'Error loading CVAE: {e}')
        cvae = None
else:
    print('CVAE weights not found. Skipping CVAE-dependent figures.')

In [ ]:
# Cell 6: Generate Synthetic Data & Run TSTR
# This cell generates synthetic spectra from the CVAE and trains the Apollo pipeline on them

if cvae is not None:
    N_SYNTHETIC_PER_CLASS = 5000  # Number of synthetic spectra per class
    num_classes = 7  # 0=Normal, 1-6=LGm subtypes
    
    tstr_results = []
    tstr_all_true = []
    tstr_all_pred = []
    
    print('Running TSTR Evaluation (Train on Synthetic, Test on Real)...')
    print('=' * 60)
    
    for fold_data in loader:
        fold_name = fold_data['fold_name']
        X_train_real, y_train_real = fold_data['train']
        X_val, y_val = fold_data['val']
        
        # Generate synthetic training data
        print(f'\n{fold_name}: Generating {N_SYNTHETIC_PER_CLASS} synthetic spectra per class...')
        
        X_syn_list = []
        y_syn_list = []
        
        unique_classes_in_fold = np.unique(y_train_real)
        
        for class_id in unique_classes_in_fold:
            # Sample from standard normal prior
            z_samples = np.random.randn(N_SYNTHETIC_PER_CLASS, 16).astype(np.float32)
            
            # One-hot label for this class
            labels = tf.keras.utils.to_categorical(
                np.full(N_SYNTHETIC_PER_CLASS, class_id), num_classes=num_classes
            )
            
            # Decode
            synthetic_spectra = cvae.decoder([z_samples, labels]).numpy()
            X_syn_list.append(synthetic_spectra[:, :, 0])  # Remove channel dim
            y_syn_list.append(np.full(N_SYNTHETIC_PER_CLASS, class_id))
        
        X_syn = np.concatenate(X_syn_list, axis=0)
        y_syn = np.concatenate(y_syn_list, axis=0)
        
        print(f'  Synthetic data: {X_syn.shape}')
        
        # Train Apollo on SYNTHETIC, test on REAL
        apollo = ApolloPipeline(n_features=20, random_state=42)
        apollo.fit(X_syn, y_syn)
        y_pred = apollo.predict(X_val)
        
        acc = accuracy_score(y_val, y_pred)
        print(f'  TSTR Accuracy: {acc:.4f}')
        
        tstr_results.append({'fold': fold_name, 'tstr_accuracy': acc})
        tstr_all_true.extend(y_val)
        tstr_all_pred.extend(y_pred)
    
    tstr_mean = np.mean([r['tstr_accuracy'] for r in tstr_results])
    tstr_std = np.std([r['tstr_accuracy'] for r in tstr_results])
    print(f'\n{"=" * 60}')
    print(f'TSTR Mean Accuracy: {tstr_mean:.4f} ± {tstr_std:.4f}')
    print(f'TRTR Mean Accuracy: {mean_acc:.4f} ± {std_acc:.4f}')
    print(f'Delta: {tstr_mean - mean_acc:+.4f}')
    
    # Save results
    tstr_df = pd.DataFrame(tstr_results)
    tstr_df.to_csv(os.path.join(PROJECT_ROOT, 'results', 'tstr_metrics.csv'), index=False)
    print('Saved TSTR metrics.')
else:
    print('Skipping TSTR (no CVAE model loaded).')

In [ ]:
# Cell 7: Figure 2 — TRTR vs TSTR Bar Chart
if cvae is not None and tstr_results:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    fold_names = [r['fold'] for r in trtr_results]
    trtr_accs = [r['accuracy'] for r in trtr_results]
    tstr_accs = [r['tstr_accuracy'] for r in tstr_results]
    
    x = np.arange(len(fold_names))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, trtr_accs, width, label='TRTR (Real)', color='#2196F3', alpha=0.85)
    bars2 = ax.bar(x + width/2, tstr_accs, width, label='TSTR (Synthetic)', color='#F44336', alpha=0.85)
    
    # Add value labels
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=9)
    
    # Chance level
    ax.axhline(1/6, color='gray', linestyle='--', alpha=0.5, label='Chance (16.7%)')
    
    ax.set_ylabel('Accuracy')
    ax.set_title('TRTR vs. TSTR: 6-Class Glioma Grading\nPatient-Level 5-Fold Cross-Validation', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f'Fold {i}' for i in range(len(fold_names))])
    ax.legend()
    ax.set_ylim(0, max(max(trtr_accs), max(tstr_accs)) + 0.1)
    ax.grid(axis='y', alpha=0.3)
    
    save_path = os.path.join(FIGURES_DIR, 'fig2_trtr_vs_tstr.pdf')
    plt.savefig(save_path)
    plt.show()
    print(f'Saved: {save_path}')
else:
    print('Skipping Figure 2 (no TSTR results).')

In [ ]:
# Cell 8: Figure 3 — t-SNE Latent Space Visualization
if cvae is not None:
    print('Generating Figure 3: t-SNE Latent Space...')
    
    # Subsample for t-SNE (50 per class)
    n_per_class = 50
    X_sub_list, y_sub_list = [], []
    
    for class_id in np.unique(loader.y_class):
        idx = np.where(loader.y_class == class_id)[0]
        n = min(n_per_class, len(idx))
        sampled = np.random.choice(idx, n, replace=False)
        X_sub_list.append(loader.X[sampled])
        y_sub_list.append(loader.y_class[sampled])
    
    X_sub = np.concatenate(X_sub_list)
    y_sub = np.concatenate(y_sub_list)
    X_sub_exp = np.expand_dims(X_sub, axis=-1)
    
    # Encode
    _, _, z_sub, _ = cvae.encoder(X_sub_exp)
    z_np = z_sub.numpy()
    
    # t-SNE
    perplexity = min(30, len(z_np) - 1)
    tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity, max_iter=1000)
    z_2d = tsne.fit_transform(z_np)
    
    # Plot
    fig, ax = plt.subplots(figsize=(9, 7))
    
    unique_classes = np.unique(y_sub)
    colors = plt.cm.Set2(np.linspace(0, 1, len(unique_classes)))
    
    for i, cls in enumerate(unique_classes):
        mask = y_sub == cls
        label = CLASS_NAMES[cls-1] if 1 <= cls <= 6 else f'Class {cls}'
        ax.scatter(z_2d[mask, 0], z_2d[mask, 1], c=[colors[i]], label=label,
                   s=40, alpha=0.7, edgecolors='black', linewidth=0.3)
    
    ax.set_title('CVAE Latent Space (t-SNE Projection)', fontweight='bold')
    ax.set_xlabel('t-SNE Dimension 1')
    ax.set_ylabel('t-SNE Dimension 2')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    save_path = os.path.join(FIGURES_DIR, 'fig3_tsne_latent.pdf')
    plt.savefig(save_path)
    plt.show()
    print(f'Saved: {save_path}')
else:
    print('Skipping Figure 3 (no CVAE).')

In [ ]:
# Cell 9: Figure 4 — Reconstruction Quality (Real vs. CVAE per class)
if cvae is not None:
    print('Generating Figure 4: Reconstruction comparison...')
    
    unique_classes = np.sort(np.unique(loader.y_class))
    # Only plot LGm-1 through LGm-6
    tumor_classes = [c for c in unique_classes if 1 <= c <= 6]
    
    n_classes = len(tumor_classes)
    fig, axes = plt.subplots(n_classes, 1, figsize=(12, 2.5 * n_classes), sharex=True)
    if n_classes == 1:
        axes = [axes]
    
    x_axis = np.arange(loader.X.shape[1])
    
    for i, cls in enumerate(tumor_classes):
        ax = axes[i]
        idx = np.where(loader.y_class == cls)[0]
        if len(idx) == 0:
            continue
        
        # Pick random sample
        sample_idx = np.random.choice(idx)
        real_spec = loader.X[sample_idx]
        real_exp = np.expand_dims(np.expand_dims(real_spec, 0), -1)
        label_oh = tf.keras.utils.to_categorical([cls], num_classes=7)
        
        recon, _, _, _ = cvae([real_exp, label_oh])
        recon_spec = recon[0, :, 0].numpy()
        
        ax.plot(x_axis, real_spec, color='black', linewidth=1, alpha=0.8, label='Real')
        ax.plot(x_axis, recon_spec, color='#D32F2F', linewidth=1, alpha=0.8, linestyle='--', label='CVAE')
        ax.set_ylabel('Intensity')
        ax.set_title(f'{CLASS_NAMES[cls-1]}', fontsize=11, fontweight='bold', loc='left')
        if i == 0:
            ax.legend(loc='upper right', framealpha=0.8)
        ax.grid(alpha=0.2)
    
    axes[-1].set_xlabel('Spectral Band Index')
    plt.suptitle('Spectral Reconstruction Quality (Real vs. CVAE)', fontweight='bold', y=1.01)
    plt.tight_layout()
    
    save_path = os.path.join(FIGURES_DIR, 'fig4_reconstruction.pdf')
    plt.savefig(save_path)
    plt.show()
    print(f'Saved: {save_path}')
else:
    print('Skipping Figure 4 (no CVAE).')

In [ ]:
# Cell 10: Figure 5 — Class Centroids: Real Mean ± σ vs. Generated at z=0
if cvae is not None:
    print('Generating Figure 5: Class Centroids with envelopes...')
    
    tumor_classes = [c for c in np.unique(loader.y_class) if 1 <= c <= 6]
    n_classes = len(tumor_classes)
    
    fig, axes = plt.subplots(n_classes, 1, figsize=(12, 2.2 * n_classes), sharex=True)
    if n_classes == 1:
        axes = [axes]
    
    x_axis = np.arange(loader.X.shape[1])
    
    for i, cls in enumerate(tumor_classes):
        ax = axes[i]
        idx = np.where(loader.y_class == cls)[0]
        
        # Real statistics
        class_data = loader.X[idx[:2000]]  # Subsample for speed
        mean_real = np.mean(class_data, axis=0)
        std_real = np.std(class_data, axis=0)
        
        # Generated at z=0
        z_zero = np.zeros((1, 16), dtype=np.float32)
        label_oh = tf.keras.utils.to_categorical([cls], num_classes=7)
        gen_spec = cvae.decoder([z_zero, label_oh])[0, :, 0].numpy()
        
        # Plot
        ax.fill_between(x_axis, mean_real - std_real, mean_real + std_real,
                        color='gray', alpha=0.25, label='Real ±1σ')
        ax.plot(x_axis, mean_real, color='black', linewidth=1.5, label='Real Mean')
        ax.plot(x_axis, gen_spec, color='#D32F2F', linewidth=1.5, linestyle='--', label='Generated (z=0)')
        ax.set_title(f'{CLASS_NAMES[cls-1]}', fontsize=11, fontweight='bold', loc='left')
        ax.set_ylabel('Intensity')
        if i == 0:
            ax.legend(loc='upper right', framealpha=0.8)
        ax.grid(alpha=0.2)
    
    axes[-1].set_xlabel('Spectral Band Index')
    plt.suptitle('Class Centroids: Real vs. Generated', fontweight='bold', y=1.01)
    plt.tight_layout()
    
    save_path = os.path.join(FIGURES_DIR, 'fig5_class_centroids.pdf')
    plt.savefig(save_path)
    plt.show()
    print(f'Saved: {save_path}')
else:
    print('Skipping Figure 5 (no CVAE).')

In [ ]:
# Cell 11: Figure 6 — Wasserstein Distance (per class, per band)
if cvae is not None:
    print('Computing Wasserstein-1 distances per class...')
    
    N_GEN = 2000  # Generate this many per class
    tumor_classes = [c for c in np.unique(loader.y_class) if 1 <= c <= 6]
    
    w1_per_class = {}
    
    for cls in tumor_classes:
        # Real data for this class
        idx = np.where(loader.y_class == cls)[0]
        real_data = loader.X[np.random.choice(idx, min(N_GEN, len(idx)), replace=False)]
        
        # Generate synthetic
        z_samples = np.random.randn(N_GEN, 16).astype(np.float32)
        labels = tf.keras.utils.to_categorical(np.full(N_GEN, cls), num_classes=7)
        syn_data = cvae.decoder([z_samples, labels]).numpy()[:, :, 0]
        
        # Compute Wasserstein-1 distance per spectral band
        n_bands = real_data.shape[1]
        w1_bands = np.zeros(n_bands)
        for b in range(n_bands):
            w1_bands[b] = wasserstein_distance(real_data[:, b], syn_data[:, b])
        
        w1_per_class[cls] = w1_bands
        print(f'  {CLASS_NAMES[cls-1]}: Mean W1 = {np.mean(w1_bands):.4f}')
    
    # Plot: Wasserstein distance per band, per class
    fig, axes = plt.subplots(len(tumor_classes), 1, figsize=(12, 2 * len(tumor_classes)), sharex=True)
    if len(tumor_classes) == 1:
        axes = [axes]
    
    for i, cls in enumerate(tumor_classes):
        ax = axes[i]
        ax.fill_between(range(len(w1_per_class[cls])), w1_per_class[cls],
                       color='#FF7043', alpha=0.7)
        ax.set_ylabel('$W_1$')
        ax.set_title(f'{CLASS_NAMES[cls-1]} (Mean: {np.mean(w1_per_class[cls]):.4f})',
                    fontsize=10, fontweight='bold', loc='left')
        ax.grid(alpha=0.2)
    
    axes[-1].set_xlabel('Spectral Band Index')
    plt.suptitle('Per-Band Wasserstein-1 Distance: Real vs. Synthetic',
                fontweight='bold', y=1.01)
    plt.tight_layout()
    
    save_path = os.path.join(FIGURES_DIR, 'fig6_wasserstein_per_band.pdf')
    plt.savefig(save_path)
    plt.show()
    print(f'Saved: {save_path}')
    
    # Summary table
    print('\n--- Wasserstein-1 Summary ---')
    for cls in tumor_classes:
        print(f'{CLASS_NAMES[cls-1]}: Mean={np.mean(w1_per_class[cls]):.4f}, '
              f'Max={np.max(w1_per_class[cls]):.4f}, '
              f'Median={np.median(w1_per_class[cls]):.4f}')
else:
    print('Skipping Figure 6 (no CVAE).')

In [ ]:
# Cell 12: Privacy Metric — Nearest Neighbor Distance (Authenticity)
if cvae is not None:
    from sklearn.neighbors import NearestNeighbors
    
    print('Computing Authenticity (Nearest Neighbor Distance)...')
    
    # Use a subset for computational feasibility
    N_EVAL = 1000
    
    for cls in tumor_classes[:3]:  # First 3 classes for demo
        idx = np.where(loader.y_class == cls)[0]
        real_sub = loader.X[np.random.choice(idx, min(N_EVAL, len(idx)), replace=False)]
        
        # Generate synthetic
        z_samples = np.random.randn(N_EVAL, 16).astype(np.float32)
        labels = tf.keras.utils.to_categorical(np.full(N_EVAL, cls), num_classes=7)
        syn_sub = cvae.decoder([z_samples, labels]).numpy()[:, :, 0]
        
        # Fit NN on real, query with synthetic
        nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
        nn.fit(real_sub)
        distances, _ = nn.kneighbors(syn_sub)
        
        print(f'  {CLASS_NAMES[cls-1]}: '
              f'Mean NN dist = {np.mean(distances):.4f}, '
              f'Min = {np.min(distances):.4f}, '
              f'Max = {np.max(distances):.4f}')
else:
    print('Skipping privacy metrics (no CVAE).')

In [ ]:
# Cell 13: Summary — Print all results for paper
print('\n' + '=' * 60)
print('PAPER RESULTS SUMMARY')
print('=' * 60)

print('\n--- Table 1: TRTR Baseline ---')
for r in trtr_results:
    print(f"  {r['fold']}: {r['accuracy']:.4f}")
print(f'  Mean ± Std: {mean_acc:.4f} ± {std_acc:.4f}')

if cvae is not None and tstr_results:
    print('\n--- Table 2: TSTR Results ---')
    for r in tstr_results:
        print(f"  {r['fold']}: {r['tstr_accuracy']:.4f}")
    print(f'  Mean ± Std: {tstr_mean:.4f} ± {tstr_std:.4f}')
    print(f'  Delta (TSTR - TRTR): {tstr_mean - mean_acc:+.4f}')

print('\n--- Figures saved to paper/figures/ ---')
for f in sorted(os.listdir(FIGURES_DIR)):
    if f.endswith(('.pdf', '.png')):
        print(f'  {f}')